# 第 18 章 · 残差连接：输出 = 输入 + 子层(输入)

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/18-residual.html) · 仓库里 `transformer-deep-dive/notes/lessons/18-residual.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 会把自己顶掉 | attention 结果 [1.46, 0.27] | — | "咬"自己那一位从 1 掉到 0.27 |
| 残差 | $x + \text{Sublayer}(x)$ | `x + sublayer(x)` | [1.46, 1.27] |
| 残差流 | $x_{12} = x_0 + s_1 + \dots + s_{12}$ | 12 次 `x = x + s` | `x − x0` 正好是 12 条批注之和 |
| 什么都不做 | 批注 = 0 | 权重全 0 的 `nn.Linear` | 不加残差输出全 0；加了输出 = 输入 |
| 叠 12 层还剩多少原文 | 传话 vs 共享文档 | 余弦相似度 | 不加残差 −0.05，加残差 0.42 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

**函数 `residual`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def residual(x: torch.Tensor, sublayer) -> torch.Tensor:
    return x + sublayer(x)

**函数 `stack`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def stack(x0: torch.Tensor, layers: list[nn.Module], use_residual: bool) -> torch.Tensor:
    x = x0
    for layer in layers:
        x = x + layer(x) if use_residual else layer(x)
    return x

**函数 `make_layers`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def make_layers(n: int, d: int, seed: int = 0) -> list[nn.Module]:
    torch.manual_seed(seed)
    return [nn.Sequential(nn.Linear(d, d), nn.Tanh()) for _ in range(n)]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
bite = torch.tensor([0.0, 1.0])
attn_out = torch.tensor([1.46, 0.27])  # 0.73 × 狗 [2, 0] + 0.27 × 咬 [0, 1]
print("'咬' =", bite.tolist(), " attention 结果 =", [round(float(v), 2) for v in attn_out])
print("加残差:", [round(float(v), 2) for v in bite + attn_out], "← 自己那一位 1 还在")

In [ ]:
print("\n残差流：x_12 = x_0 + 12 条批注")
torch.manual_seed(0)
x0 = torch.randn(4)
notes = [torch.randn(4) * 0.1 for _ in range(12)]
x = x0.clone()
for s in notes:
    x = x + s
print("  x_12 − x_0 == 12 条批注之和:", torch.allclose(x - x0, torch.stack(notes).sum(0)))

In [ ]:
print("\n'什么都不做'：子层权重全是 0（批注 = 0）")
quiet = nn.Linear(4, 4)
nn.init.zeros_(quiet.weight)
nn.init.zeros_(quiet.bias)
print("  不加残差: 输出", quiet(x0).tolist(), "← 原文没了")
print("  加残差  : 输出 == 输入", torch.equal(residual(x0, quiet), x0))

In [ ]:
print("\n叠 12 个随机子层，最后还剩多少原文（和 x_0 的余弦相似度，1 = 方向完全一样）")
layers = make_layers(12, 64)
x0 = torch.randn(64)
with torch.no_grad():
    for flag, name in [(False, "传话游戏（不加残差）"), (True, "共享文档（加残差）")]:
        out = stack(x0, layers, flag)
        print(f"  {name}: {float(F.cosine_similarity(out, x0, dim=0)):.2f}")

## 3. 代码里才看得到的

1. **余弦相似度**（cosine similarity：两个向量方向有多像，1 = 同方向，0 = 不相干，负数 = 反方向）：叠 12 个随机子层后，不加残差的输出和原文几乎没关系（−0.05）；加了残差还留着一大块（0.42）。不是 1，因为 12 条批注也加进去了。
2. **`nn.init.zeros_`**：把参数直接填成 0（末尾的下划线表示"原地改"）。批注为 0 时，残差版的输出和输入一位都不差。

## 4. 自己改一改

- 把上面任意一格里的数字改一改，再按 Shift + Enter，看结果怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 子层只写批注，不撕原文：输出 = x + 子层(x)；
> 层层往上加，原词一直在；
> 要相加就得一样长，没话说就写 0，原样透传。